Author: Konstantin Malanchev and the LINCC Frameworks team

Last updated: November 2026

# Homework 1b: Palomar 12 color-magnitude diagram

**Task:** plot a color-magnitude diagram of the globular cluster Palomar 12.

**Plan:**
1. Select a small cone around the cluster with `ConeSearch`
2. Select point sources with good photometry
3. Plot a color-magnitude diagram

If you'd like to solve it yourself, `Notebook_1a_Intro` may help.

> [!NOTE]
> This notebook uses Rubin DP2 data, so it can be run by Rubin data rights holders (DRHs) only, e.g. on the [Rubin Science Platform](https://data.lsst.cloud) with the latest weekly image.
> Since we would like to share the notebook and its outputs with everyone, we never print or plot individual rows of the DP2 catalog: all the outputs are either catalog metadata or aggregated statistics.

## Solution

In [ ]:
# lsdb is pre-installed on RSP, uncomment the line below if you are on another platform
# %pip install -qq lsdb

In [ ]:
# Standard library imports
import logging
import os
import warnings

# Third-party imports
import dask
import lsdb
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dask.distributed import Client
from lsdb import ConeSearch
from matplotlib import rcParams

# Jupyter magic commands
%matplotlib inline

# Configuration
warnings.simplefilter("ignore")
logging.getLogger("distributed").setLevel(logging.WARNING)
rcParams["savefig.dpi"] = 150
rcParams["font.size"] = 14
plt.rc("font", family="serif")
mpl.rcParams["axes.linewidth"] = 1.5
dask.config.set({"dataframe.convert-string": False})

## Setup Dask cluster

We use the same Dask setup as in `Notebook_1a_Intro`: sixteen workers, one per CPU of a "Large" RSP container, with one thread and 2 GB of memory each, which is enough for this low-memory pipeline, and `/deleted-sundays/` for spilling data to disk.

In [ ]:
client = Client(
    n_workers=16,
    threads_per_worker=1,
    local_directory=f"/deleted-sundays/{os.environ.get('USER', 'dask_scratch')}",
)

# Print the dashboard link and port
print(f"Dask is running at: {client.dashboard_link}")

### Select the cluster

We select a 0.5-degree cone around the cluster center with `lsdb.ConeSearch`. The coordinates are from the [Harris (1996, 2010 edition)](https://physics.mcmaster.ca/~harris/mwgc.dat) catalog of Milky Way globular clusters.

In [ ]:
# 0.5-degree cone around Palomar 12
cone_search = ConeSearch(ra=326.66, dec=-21.25, radius_arcsec=0.5 * 3600)

# DP2 Object collection on RSP, check https://lsdb.io/dp2 for the paths on other platforms
object_cluster = lsdb.open_catalog(
    "/rubin/lsdb_data/dp2/object_collection",
    # coordinates and _healpix_29 are always loaded
    columns=["objectId", "refExtendedness", "r_psfMag", "r_psfMagErr", "z_psfMag", "z_psfMagErr"],
    search_filter=cone_search,
)
object_cluster

### Select point sources

As in `Notebook_1a_Intro`, we select point-like objects, but with a less strict cut on PSF magnitude errors: smaller than 0.1 mag in r and z bands.

In [ ]:
stars = object_cluster.query("refExtendedness == 0 and r_psfMagErr < 0.1 and z_psfMagErr < 0.1")
stars

### Color-magnitude diagram

We plot r magnitude vs r−z color as a 2D histogram, also known as a Hess diagram. As in `Notebook_1a_Intro`, we compute partial histograms with `map_partitions()` and sum them, so only bin counts leave the workers.

In [ ]:
RZ_BINS = np.linspace(-0.5, 2.5, 61)
R_BINS = np.linspace(16.0, 25.0, 91)


def color_magnitude_hist(df):
    """Partial 2D histogram of r-z vs r for a single partition."""
    counts, _, _ = np.histogram2d(df["r_psfMag"] - df["z_psfMag"], df["r_psfMag"], bins=[RZ_BINS, R_BINS])
    rz_idx, r_idx = np.nonzero(counts)
    return pd.DataFrame({"rz_bin": rz_idx, "r_bin": r_idx, "count": counts[rz_idx, r_idx].astype(np.int64)})


partial_hists = stars.map_partitions(color_magnitude_hist, meta={"rz_bin": int, "r_bin": int, "count": int})

In [ ]:
partial_hists_df = partial_hists.compute()

hist = np.zeros((len(RZ_BINS) - 1, len(R_BINS) - 1), dtype=np.int64)
np.add.at(hist, (partial_hists_df["rz_bin"], partial_hists_df["r_bin"]), partial_hists_df["count"])

# A bin with a single object exposes its photometry up to the bin size, so we remove such bins
hist[hist == 1] = 0
print(f"Total number in the histogram: {hist.sum():,}")

In [ ]:
# Empty bins are NaN, so they are not colored
hist_plot = np.where(hist > 0, hist, np.nan)

fig, ax = plt.subplots(figsize=(8, 10))
mesh = ax.pcolormesh(RZ_BINS, R_BINS, hist_plot.T, cmap="viridis")
fig.colorbar(mesh, ax=ax, label="Number")
ax.invert_yaxis()
ax.set_xlabel("$r - z$ [mag]")
ax.set_ylabel("$r$ [mag]")
ax.set_title("Rubin DP2 point sources around Palomar 12")
plt.tight_layout()
plt.show()

In [ ]:
client.close()

## Citing

If you use the tools or the data from this notebook in your research, please cite the following papers.

<details>
<summary>For LSDB and HATS cite: Caplar et al. (2025), arXiv:2501.02103</summary>

```bibtex
% ADS bibcode: 2025arXiv250102103C
@ARTICLE{Caplar_LSDB_2025,
       author = {{Caplar}, Neven and {Beebe}, Wilson and {Branton}, Doug and {Campos}, Sandro and {Connolly}, Andrew and {DeLucchi}, Melissa and {Jones}, Derek and {Juric}, Mario and {Kubica}, Jeremy and {Malanchev}, Konstantin and {Mandelbaum}, Rachel and {McGuire}, Sean},
        title = "{Using LSDB to enable large-scale catalog distribution, cross-matching, and analytics}",
      journal = {arXiv e-prints},
     keywords = {Instrumentation and Methods for Astrophysics},
         year = 2025,
        month = jan,
          eid = {arXiv:2501.02103},
        pages = {arXiv:2501.02103},
          doi = {10.48550/arXiv.2501.02103},
archivePrefix = {arXiv},
       eprint = {2501.02103},
 primaryClass = {astro-ph.IM},
       adsurl = {https://ui.adsabs.harvard.edu/abs/2025arXiv250102103C},
      adsnote = {Provided by the SAO/NASA Astrophysics Data System}
}
```

</details>

<details>
<summary>For Rubin catalogs in HATS cite: Malanchev et al. (2025), arXiv:2506.23955</summary>

```bibtex
% ADS bibcode: 2025arXiv250623955M
@ARTICLE{Malanchev_DP1_2025,
       author = {{Malanchev}, Konstantin and {DeLucchi}, Melissa and {Caplar}, Neven and {Malz}, Alex I. and {Alexov}, Anastasia and {Aubourg}, Eric and {E Bauer}, Amanda and {Beebe}, Wilson and {Bellm}, Eric C. and {Blum}, Robert David and {Branton}, Doug and {Campos}, Sandro and {Calabrese}, Daniel and {Carlin}, Jeffrey L. and {Choi}, Yumi and {Connolly}, Andrew and {Dai}, Mi and {Daly}, Philip N. and {Daruich}, Felipe and {Daubard}, Guillaume and {Delgado}, Francisco and {Drass}, Holger and {Fonseca Alvarez}, Gloria and {Gangler}, Emmanuel and {Guy}, Leanne P. and {Ivezi{\'c}}, {\v{Z}}eljko and {Jammes}, Fabrice and {Jannuzi}, Buell T. and {Jenness}, Tim and {Jimenez}, David and {Jones}, Derek T. and {Juri{\'c}}, Mario and {Kahn}, Steven M. and {Kang}, Yijung and {Kannawadi}, Arun and {Kubica}, Jeremy and {Le Guillou}, Laurent and {Lynn}, Olivia and {Mandelbaum}, Rachel and {Moniez}, Marc and {Morrison}, Christopher B. and {McGuire}, Sean and {Ordenes}, Ian E. and {Patterson}, Maria T. and {Plazas Malag{\'o}n}, Andr{\'e}s A. and {Qiu}, Yongqiang and {Quint}, Bruno and {Radeka}, Veljko and {van Reeven}, Wouter and {Riot}, Vincent J. and {S{\'a}nchez}, Bruno O. and {Sanmartim}, David and {Sebag}, Jacques and {Shugart}, Alysha and {Sullivan}, Ian S. and {Taranu}, Dan S. and {Urbach}, Elana K. and {Wood-Vasey}, W.~M. and {Zhang}, Tianqing},
        title = "{Variability-finding in Rubin Data Preview 1 with LSDB}",
      journal = {arXiv e-prints},
     keywords = {Instrumentation and Methods for Astrophysics, Solar and Stellar Astrophysics},
         year = 2025,
        month = jun,
          eid = {arXiv:2506.23955},
        pages = {arXiv:2506.23955},
          doi = {10.48550/arXiv.2506.23955},
archivePrefix = {arXiv},
       eprint = {2506.23955},
 primaryClass = {astro-ph.IM},
       adsurl = {https://ui.adsabs.harvard.edu/abs/2025arXiv250623955M},
      adsnote = {Provided by the SAO/NASA Astrophysics Data System}
}
```

</details>

<details>
<summary>For Rubin DP2 data cite: Rubin Observatory (2026), RTN-115, doi:10.71929/rubin/3377440; DP2 dataset, doi:10.71929/rubin/3382528; DP2 Object catalog, doi:10.71929/rubin/3382540</summary>

```bibtex
% ADS bibcode: 2026rubn.rept....2V
@TECHREPORT{Rubin_DP2_2026,
       author = {{Vera C. Rubin Observatory Team}},
        title = "{RTN-115: The Vera C. Rubin Observatory Data Preview 2}",
  institution = {NSF-DOE Vera C. Rubin Observatory},
     keywords = {79 ASTRONOMY AND ASTROPHYSICS},
         year = 2026,
 howpublished = {NSF-DOE Vera C. Rubin Observatory Technical Report},
          doi = {10.71929/RUBIN/3377440},
       adsurl = {https://ui.adsabs.harvard.edu/abs/2026rubn.rept....2V},
      adsnote = {Provided by the SAO/NASA Astrophysics Data System}
}

% ADS bibcode: 2026lsst.data...42N
@dataset{Rubin_DP2Data_2026,
       author = {{NSF-DOE Vera C. Rubin Observatory}},
        title = "{Legacy Survey of Space and Time Data Preview 2}",
 howpublished = {NSF-DOE Vera C. Rubin Observatory Data Set},
         year = 2026,
        month = jan,
          doi = {10.71929/RUBIN/3382528},
    publisher = {Rubin Obs},
       adsurl = {https://ui.adsabs.harvard.edu/abs/2026lsst.data...42N},
      adsnote = {Provided by the SAO/NASA Astrophysics Data System}
}

% ADS bibcode: 2026lsst.data...43N
@dataset{Rubin_DP2Object_2026,
       author = {{NSF-DOE Vera C. Rubin Observatory}},
        title = "{Legacy Survey of Space and Time Data Preview 2: Object searchable catalog}",
 howpublished = {NSF-DOE Vera C. Rubin Observatory Data Set},
         year = 2026,
        month = jan,
          doi = {10.71929/RUBIN/3382540},
    publisher = {Rubin Obs},
       adsurl = {https://ui.adsabs.harvard.edu/abs/2026lsst.data...43N},
      adsnote = {Provided by the SAO/NASA Astrophysics Data System}
}
```

</details>